### Import libraries

In [3]:
from pathlib import Path
import pandas as pd

### Load the dataset

In [4]:
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

source = ROOT / "datasets/processed/Combaine_paddy.csv"

if not source.exists():
    raise FileNotFoundError("Run notebook 01 first.")

df = pd.read_csv(
    source,
    keep_default_na=False,
    encoding="utf-8-sig"
)

### deatils of dataset

In [6]:
df.shape

(25440, 16)

In [8]:
print("Columns:", df.columns.tolist())

Columns: ['qa_id', 'pair_id', 'question', 'answer', 'language', 'category_en', 'category_si', 'issue_id', 'issue_en', 'crop_stage_en', 'crop_stage_si', 'urgency', 'source_url', 'data_origin', 'review_status', 'is_real_farmer_query']


In [9]:
print(df.isna().sum().sort_values(ascending=False))

qa_id                   0
pair_id                 0
question                0
answer                  0
language                0
category_en             0
category_si             0
issue_id                0
issue_en                0
crop_stage_en           0
crop_stage_si           0
urgency                 0
source_url              0
data_origin             0
review_status           0
is_real_farmer_query    0
dtype: int64


### dataset clean

In [ ]:
for col in [
    "question",
    "answer",
    "language",
    "category_en",
    "issue_id",
    "pair_id"
]:
    assert col in df.columns, f"Required column missing: {col}"

In [10]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()

C:\Users\USER\AppData\Local\Temp\ipykernel_7976\1898379864.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


In [11]:
before = len(df)

df = df.drop_duplicates(
    subset=["pair_id", "language"],
    keep="first"
).copy()

print(
    "Rows removed as duplicate pair/language keys:",
    before - len(df)
)

Rows removed as duplicate pair/language keys: 0


In [13]:
assert (df["question"].str.len() > 0).all()
assert (df["answer"].str.len() > 0).all()
assert set(df["language"].unique()) == {"si-LK", "en-LK"}
assert df.groupby("issue_id")["category_en"].nunique().max() == 1, "An issue_id maps to multiple categories. Review labels."

### Save the cleaned dataset

In [15]:
out = ROOT / "datasets/processed/cleaned_paddy.csv"

df.to_csv(
    out,
    index=False,
    encoding="utf-8-sig"
)